# Conditional RNN-Diffusion Forecasting


In [ ]:
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", message=r".*'H' is deprecated.*", category=FutureWarning)

HOUSEHOLD_ID_COL = "CUSTOMER_KEY"
TIME_COL = "READING_DATETIME"
TARGET_COL = "kWh"

# Manual window length settings.
# Set CONTEXT_LEN to 0 for a no-context run.
# Use the context value shown below for the matching context experiment.
#
# Frequency | 24h target | 7d target | 28d target | 2d context | 7d context
# 30min     | 48         | 336       | 1344       | 96         | 336
# 1H        | 24         | 168       | 672        | 48         | 168
# 2H        | 12         | 84        | 336        | 24         | 84
# 3H        | 8          | 56        | 224        | 16         | 56
FREQ = "30min"
PRIMARY_HORIZON = "24h"
SEQ_LEN = 48
CONTEXT_LEN = 96

# Final experiments advanced by one complete forecast horizon.
WINDOW_STRIDE = SEQ_LEN

HAS_CONTEXT = CONTEXT_LEN > 0
MODEL_LABEL = 'RNN-Diffusion-Forecaster' if HAS_CONTEXT else 'RNN-Diffusion-NoContext-Forecaster'
MODEL_TAG = 'forecast-rnn-diffusion' if HAS_CONTEXT else 'forecast-nocontext-rnn-diffusion'
MODEL_KIND = 'rnn_diffusion'
LOSS_MODE = "core" if HAS_CONTEXT else "nocontext_core"

SEED = 0
VAL_FRAC = 0.20
BATCH_SIZE = 64
LR = 2e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hidden_dim = 128
cond_dim = 32
grad_clip = 1.0
sample_temperature = 0.8

epochs_diff = 500
diffusion_steps = 300
beta_start = 1e-4
beta_end = 0.02

STATIC_COLS = [
    "NUM_OCCUPANTS", "NUM_ROOMS_HEATED", "NUM_REFRIGERATORS",
    "Unit", "SemiDetached", "SeparateHouse",
    "HAS_GAS_HEATING", "HAS_GAS_HOT_WATER", "HAS_GAS_COOKING",
    "HAS_POOLPUMP", "Ducted", "SplitSystem", "NoAirCon", "OtherAirCon",
    "CONTROLLED_LOAD_CNT",
]

CATEGORICAL_COLS = ["StationNo", "TRIAL_REGION_NAME"]

# Past covariates contain calendar position and measurements already observed.
PAST_COV_COLS = [
    "Hour_sin", "Hour_cos",
    "Weekday_sin", "Weekday_cos",
    "Month_sin", "Month_cos",
    "Temperature", "CDD", "HDD", "wind_speed",
    "Temperature_lag_2", "Temperature_lag_6",
    "Temperature_lag_48", "Temperature_lag_144",
    "temperature_lag_2", "temperature_lag_6",
    "temperature_lag_48", "temperature_lag_144",
]

# Future inputs are restricted to calendar values known before prediction.
FUTURE_COV_COLS = [
    "Hour_sin", "Hour_cos",
    "Weekday_sin", "Weekday_cos",
    "Month_sin", "Month_cos",
]

RAW_TIME_COV_COLS = [
    "Temperature", "Weekday", "Month", "Hour", "CDD", "HDD", "wind_speed",
    "Temperature_lag_2", "Temperature_lag_6",
    "Temperature_lag_48", "Temperature_lag_144",
    "temperature_lag_2", "temperature_lag_6",
    "temperature_lag_48", "temperature_lag_144",
]

def pandas_freq(freq):
    return freq.replace("H", "h")


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("medium")

print(
    f"{MODEL_LABEL} | FREQ={FREQ} | horizon={PRIMARY_HORIZON} | "
    f"context={CONTEXT_LEN} | horizon_steps={SEQ_LEN} | "
    f"stride={WINDOW_STRIDE} | device={DEVICE}"
)


### Preprocessing Roadmap

The preprocessing stage is intentionally a short pipeline rather than a model-specific trick:

1. Load the shared household energy table.
2. Keep the columns needed for this experiment and fill any missing optional covariates.
3. Convert timestamps, customer IDs, numeric covariates, and categorical IDs into model-ready types.
4. Add cyclical calendar features so hour, weekday, and month wrap around naturally.
5. Resample each customer to the selected frequency.
6. Split by customer so validation households are unseen during training.
7. Build context/target windows or library time-series objects, then scale the model inputs.

The small checks in this section are kept only where they prevent silent data leakage, missing-column errors, or invalid tensor shapes.


In [ ]:
# Place the dataset in the sorted directory, one level above this model notebook.
DATA_PATH = Path("..") / "data_with_weather.pickle"
df = pd.read_pickle(DATA_PATH)
print("Data file:", DATA_PATH)

# Cyclical sin/cos encoding preserves periodic hour/weekday/month distance; source: https://feature-engine.trainindata.com/en/latest/user_guide/creation/CyclicalFeatures.html
def add_cyclical_time_features(frame, datetime_values):
    dt = pd.DatetimeIndex(pd.to_datetime(datetime_values))
    hour = dt.hour.astype(np.float32)
    minute = dt.minute.astype(np.float32)
    hour_float = hour + minute / 60.0
    weekday = dt.weekday.astype(np.float32)
    month = dt.month.astype(np.float32)

    frame["Hour_sin"] = np.sin(2 * np.pi * hour_float / 24.0).astype("float32")
    frame["Hour_cos"] = np.cos(2 * np.pi * hour_float / 24.0).astype("float32")
    frame["Weekday_sin"] = np.sin(2 * np.pi * weekday / 7.0).astype("float32")
    frame["Weekday_cos"] = np.cos(2 * np.pi * weekday / 7.0).astype("float32")
    frame["Month_sin"] = np.sin(2 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Month_cos"] = np.cos(2 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Hour"] = hour.astype("float32")
    frame["Weekday"] = weekday.astype("float32")
    frame["Month"] = month.astype("float32")
    return frame

# Keep only expected columns. Missing optional covariates are created as zeros so the notebook shape stays stable.
if "READING_DATETIME" not in df.columns: df["READING_DATETIME"] = 0.0
if "kWh" not in df.columns: df["kWh"] = 0.0
if "CUSTOMER_KEY" not in df.columns: df["CUSTOMER_KEY"] = 0.0
if "NUM_OCCUPANTS" not in df.columns: df["NUM_OCCUPANTS"] = 0.0
if "NUM_ROOMS_HEATED" not in df.columns: df["NUM_ROOMS_HEATED"] = 0.0
if "NUM_REFRIGERATORS" not in df.columns: df["NUM_REFRIGERATORS"] = 0.0
if "Unit" not in df.columns: df["Unit"] = 0.0
if "SemiDetached" not in df.columns: df["SemiDetached"] = 0.0
if "SeparateHouse" not in df.columns: df["SeparateHouse"] = 0.0
if "HAS_GAS_HEATING" not in df.columns: df["HAS_GAS_HEATING"] = 0.0
if "HAS_GAS_HOT_WATER" not in df.columns: df["HAS_GAS_HOT_WATER"] = 0.0
if "HAS_GAS_COOKING" not in df.columns: df["HAS_GAS_COOKING"] = 0.0
if "HAS_POOLPUMP" not in df.columns: df["HAS_POOLPUMP"] = 0.0
if "Ducted" not in df.columns: df["Ducted"] = 0.0
if "SplitSystem" not in df.columns: df["SplitSystem"] = 0.0
if "NoAirCon" not in df.columns: df["NoAirCon"] = 0.0
if "OtherAirCon" not in df.columns: df["OtherAirCon"] = 0.0
if "CONTROLLED_LOAD_CNT" not in df.columns: df["CONTROLLED_LOAD_CNT"] = 0.0
if "StationNo" not in df.columns: df["StationNo"] = "missing"
if "TRIAL_REGION_NAME" not in df.columns: df["TRIAL_REGION_NAME"] = "missing"
if "Temperature" not in df.columns: df["Temperature"] = 0.0
temperature_for_degree_days = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0)
if "Weekday" not in df.columns: df["Weekday"] = 0.0
if "Month" not in df.columns: df["Month"] = 0.0
if "Hour" not in df.columns: df["Hour"] = 0.0
if "CDD" not in df.columns: df["CDD"] = np.maximum(temperature_for_degree_days - 18.0, 0.0).astype("float32")
if "HDD" not in df.columns: df["HDD"] = np.maximum(18.0 - temperature_for_degree_days, 0.0).astype("float32")
if "wind_speed" not in df.columns: df["wind_speed"] = 0.0
if "Temperature_lag_2" not in df.columns: df["Temperature_lag_2"] = 0.0
if "Temperature_lag_6" not in df.columns: df["Temperature_lag_6"] = 0.0
if "Temperature_lag_48" not in df.columns: df["Temperature_lag_48"] = 0.0
if "Temperature_lag_144" not in df.columns: df["Temperature_lag_144"] = 0.0
if "temperature_lag_2" not in df.columns: df["temperature_lag_2"] = 0.0
if "temperature_lag_6" not in df.columns: df["temperature_lag_6"] = 0.0
if "temperature_lag_48" not in df.columns: df["temperature_lag_48"] = 0.0
if "temperature_lag_144" not in df.columns: df["temperature_lag_144"] = 0.0
needed = [TIME_COL, TARGET_COL, HOUSEHOLD_ID_COL] + STATIC_COLS + CATEGORICAL_COLS + RAW_TIME_COV_COLS
df = df[needed].copy()

# Convert identifiers and timestamps before sorting.
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df[HOUSEHOLD_ID_COL] = pd.to_numeric(df[HOUSEHOLD_ID_COL], errors="coerce").astype("int64")
df = df.sort_values([HOUSEHOLD_ID_COL, TIME_COL])

# Convert each continuous covariate explicitly so the preprocessing reads column-by-column.
df["NUM_OCCUPANTS"] = pd.to_numeric(df["NUM_OCCUPANTS"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_ROOMS_HEATED"] = pd.to_numeric(df["NUM_ROOMS_HEATED"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_REFRIGERATORS"] = pd.to_numeric(df["NUM_REFRIGERATORS"], errors="coerce").fillna(0.0).astype("float32")
df["Unit"] = pd.to_numeric(df["Unit"], errors="coerce").fillna(0.0).astype("float32")
df["SemiDetached"] = pd.to_numeric(df["SemiDetached"], errors="coerce").fillna(0.0).astype("float32")
df["SeparateHouse"] = pd.to_numeric(df["SeparateHouse"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HEATING"] = pd.to_numeric(df["HAS_GAS_HEATING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HOT_WATER"] = pd.to_numeric(df["HAS_GAS_HOT_WATER"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_COOKING"] = pd.to_numeric(df["HAS_GAS_COOKING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_POOLPUMP"] = pd.to_numeric(df["HAS_POOLPUMP"], errors="coerce").fillna(0.0).astype("float32")
df["Ducted"] = pd.to_numeric(df["Ducted"], errors="coerce").fillna(0.0).astype("float32")
df["SplitSystem"] = pd.to_numeric(df["SplitSystem"], errors="coerce").fillna(0.0).astype("float32")
df["NoAirCon"] = pd.to_numeric(df["NoAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["OtherAirCon"] = pd.to_numeric(df["OtherAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["CONTROLLED_LOAD_CNT"] = pd.to_numeric(df["CONTROLLED_LOAD_CNT"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0).astype("float32")
df["Weekday"] = pd.to_numeric(df["Weekday"], errors="coerce").fillna(0.0).astype("float32")
df["Month"] = pd.to_numeric(df["Month"], errors="coerce").fillna(0.0).astype("float32")
df["Hour"] = pd.to_numeric(df["Hour"], errors="coerce").fillna(0.0).astype("float32")
df["CDD"] = pd.to_numeric(df["CDD"], errors="coerce").fillna(0.0).astype("float32")
df["HDD"] = pd.to_numeric(df["HDD"], errors="coerce").fillna(0.0).astype("float32")
df["wind_speed"] = pd.to_numeric(df["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_2"] = pd.to_numeric(df["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_6"] = pd.to_numeric(df["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_48"] = pd.to_numeric(df["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_144"] = pd.to_numeric(df["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_2"] = pd.to_numeric(df["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_6"] = pd.to_numeric(df["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_48"] = pd.to_numeric(df["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_144"] = pd.to_numeric(df["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["kWh"] = pd.to_numeric(df["kWh"], errors="coerce").fillna(0.0).astype("float32")

# Encode the two static categorical fields as integer IDs for embedding layers.
df["StationNo"] = df["StationNo"].astype("string").fillna("missing")
df["StationNo_id"], _ = pd.factorize(df["StationNo"], sort=True)
df["StationNo_id"] = df["StationNo_id"].astype("int64")
df["TRIAL_REGION_NAME"] = df["TRIAL_REGION_NAME"].astype("string").fillna("missing")
df["TRIAL_REGION_NAME_id"], _ = pd.factorize(df["TRIAL_REGION_NAME"], sort=True)
df["TRIAL_REGION_NAME_id"] = df["TRIAL_REGION_NAME_id"].astype("int64")

num_stations = int(df["StationNo_id"].max()) + 1
num_regions = int(df["TRIAL_REGION_NAME_id"].max()) + 1

# Resample each customer to FREQ. kWh is summed over the bin; static fields use the first value; weather uses the mean.
agg = {
    TARGET_COL: "sum",
    "NUM_OCCUPANTS": "first",
    "NUM_ROOMS_HEATED": "first",
    "NUM_REFRIGERATORS": "first",
    "Unit": "first",
    "SemiDetached": "first",
    "SeparateHouse": "first",
    "HAS_GAS_HEATING": "first",
    "HAS_GAS_HOT_WATER": "first",
    "HAS_GAS_COOKING": "first",
    "HAS_POOLPUMP": "first",
    "Ducted": "first",
    "SplitSystem": "first",
    "NoAirCon": "first",
    "OtherAirCon": "first",
    "CONTROLLED_LOAD_CNT": "first",
    "StationNo_id": "first",
    "TRIAL_REGION_NAME_id": "first",
    "Temperature": "mean",
    "Weekday": "mean",
    "Month": "mean",
    "Hour": "mean",
    "CDD": "mean",
    "HDD": "mean",
    "wind_speed": "mean",
    "Temperature_lag_2": "mean",
    "Temperature_lag_6": "mean",
    "Temperature_lag_48": "mean",
    "Temperature_lag_144": "mean",
    "temperature_lag_2": "mean",
    "temperature_lag_6": "mean",
    "temperature_lag_48": "mean",
    "temperature_lag_144": "mean",
}
df = (
    df.set_index(TIME_COL)
      .groupby(HOUSEHOLD_ID_COL)
      .resample(pandas_freq(FREQ))
      .agg(agg)
      .reset_index()
      .sort_values([HOUSEHOLD_ID_COL, TIME_COL])
)

# Final numeric cleanup after resampling.
df["NUM_OCCUPANTS"] = pd.to_numeric(df["NUM_OCCUPANTS"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_ROOMS_HEATED"] = pd.to_numeric(df["NUM_ROOMS_HEATED"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_REFRIGERATORS"] = pd.to_numeric(df["NUM_REFRIGERATORS"], errors="coerce").fillna(0.0).astype("float32")
df["Unit"] = pd.to_numeric(df["Unit"], errors="coerce").fillna(0.0).astype("float32")
df["SemiDetached"] = pd.to_numeric(df["SemiDetached"], errors="coerce").fillna(0.0).astype("float32")
df["SeparateHouse"] = pd.to_numeric(df["SeparateHouse"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HEATING"] = pd.to_numeric(df["HAS_GAS_HEATING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HOT_WATER"] = pd.to_numeric(df["HAS_GAS_HOT_WATER"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_COOKING"] = pd.to_numeric(df["HAS_GAS_COOKING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_POOLPUMP"] = pd.to_numeric(df["HAS_POOLPUMP"], errors="coerce").fillna(0.0).astype("float32")
df["Ducted"] = pd.to_numeric(df["Ducted"], errors="coerce").fillna(0.0).astype("float32")
df["SplitSystem"] = pd.to_numeric(df["SplitSystem"], errors="coerce").fillna(0.0).astype("float32")
df["NoAirCon"] = pd.to_numeric(df["NoAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["OtherAirCon"] = pd.to_numeric(df["OtherAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["CONTROLLED_LOAD_CNT"] = pd.to_numeric(df["CONTROLLED_LOAD_CNT"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0).astype("float32")
df["CDD"] = pd.to_numeric(df["CDD"], errors="coerce").fillna(0.0).astype("float32")
df["HDD"] = pd.to_numeric(df["HDD"], errors="coerce").fillna(0.0).astype("float32")
df["wind_speed"] = pd.to_numeric(df["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_2"] = pd.to_numeric(df["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_6"] = pd.to_numeric(df["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_48"] = pd.to_numeric(df["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_144"] = pd.to_numeric(df["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_2"] = pd.to_numeric(df["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_6"] = pd.to_numeric(df["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_48"] = pd.to_numeric(df["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_144"] = pd.to_numeric(df["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["kWh"] = pd.to_numeric(df["kWh"], errors="coerce").fillna(0.0).astype("float32")
df["StationNo_id"] = pd.to_numeric(df["StationNo_id"], errors="coerce").ffill().bfill().fillna(0).astype("int64")
df["TRIAL_REGION_NAME_id"] = pd.to_numeric(df["TRIAL_REGION_NAME_id"], errors="coerce").ffill().bfill().fillna(0).astype("int64")

print("Preprocessed dataframe shape:", df.shape)
print("Static category counts:", "stations", num_stations, "| regions", num_regions)


In [ ]:
# Use a seeded random generator so the customer split is repeatable.
rng = np.random.default_rng(SEED)
# Split by unique customers, not by rows/windows, to avoid leakage.
all_customers = np.array(sorted(df[HOUSEHOLD_ID_COL].dropna().unique()))
n_val = max(1, int(len(all_customers) * VAL_FRAC))
val_customers = set(rng.choice(all_customers, size=n_val, replace=False).tolist())
print(f"Customers total={len(all_customers)} | train={len(all_customers)-len(val_customers)} | val={len(val_customers)}")

def empty_float(*shape):
    return np.empty(shape, dtype=np.float32)

def empty_int(*shape):
    return np.empty(shape, dtype=np.int64)

# Accumulate train and validation windows before stacking them into tensors.
y_past_parts, y_future_parts = [], []
x_time_past_parts, x_time_future_parts = [], []
x_static_parts, x_static_cat_parts = [], []
y_past_val_parts, y_future_val_parts = [], []
x_time_past_val_parts, x_time_future_val_parts = [], []
x_static_val_parts, x_static_cat_val_parts = [], []

# Process one customer at a time so windows never cross household boundaries.
for household_id, g in df.groupby(HOUSEHOLD_ID_COL):
    g = g.sort_values(TIME_COL).drop_duplicates(subset=[TIME_COL], keep="last")
    if g.empty:
        continue

    # Rebuild a complete regular timestamp index for this customer.
    idx = pd.date_range(g[TIME_COL].min(), g[TIME_COL].max(), freq=pandas_freq(FREQ))
    g = g.set_index(TIME_COL).reindex(idx)
    g.index.name = TIME_COL
    g[HOUSEHOLD_ID_COL] = household_id

    # Fill static customer fields explicitly. These should be constant within a household.
    g["NUM_OCCUPANTS"] = pd.to_numeric(g["NUM_OCCUPANTS"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["NUM_ROOMS_HEATED"] = pd.to_numeric(g["NUM_ROOMS_HEATED"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["NUM_REFRIGERATORS"] = pd.to_numeric(g["NUM_REFRIGERATORS"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["Unit"] = pd.to_numeric(g["Unit"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["SemiDetached"] = pd.to_numeric(g["SemiDetached"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["SeparateHouse"] = pd.to_numeric(g["SeparateHouse"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["HAS_GAS_HEATING"] = pd.to_numeric(g["HAS_GAS_HEATING"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["HAS_GAS_HOT_WATER"] = pd.to_numeric(g["HAS_GAS_HOT_WATER"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["HAS_GAS_COOKING"] = pd.to_numeric(g["HAS_GAS_COOKING"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["HAS_POOLPUMP"] = pd.to_numeric(g["HAS_POOLPUMP"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["Ducted"] = pd.to_numeric(g["Ducted"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["SplitSystem"] = pd.to_numeric(g["SplitSystem"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["NoAirCon"] = pd.to_numeric(g["NoAirCon"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["OtherAirCon"] = pd.to_numeric(g["OtherAirCon"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["CONTROLLED_LOAD_CNT"] = pd.to_numeric(g["CONTROLLED_LOAD_CNT"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    g["StationNo_id"] = pd.to_numeric(g["StationNo_id"], errors="coerce").ffill().bfill().fillna(0).astype("int64")
    g["TRIAL_REGION_NAME_id"] = pd.to_numeric(g["TRIAL_REGION_NAME_id"], errors="coerce").ffill().bfill().fillna(0).astype("int64")

    # Add cyclical features once on the final per-customer timestamp grid.
    add_cyclical_time_features(g, g.index)

    # Fill target and time-varying covariates explicitly.
    g["kWh"] = pd.to_numeric(g["kWh"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature"] = pd.to_numeric(g["Temperature"], errors="coerce").fillna(0.0).astype("float32")
    g["Hour_sin"] = pd.to_numeric(g["Hour_sin"], errors="coerce").fillna(0.0).astype("float32")
    g["Hour_cos"] = pd.to_numeric(g["Hour_cos"], errors="coerce").fillna(0.0).astype("float32")
    g["Weekday_sin"] = pd.to_numeric(g["Weekday_sin"], errors="coerce").fillna(0.0).astype("float32")
    g["Weekday_cos"] = pd.to_numeric(g["Weekday_cos"], errors="coerce").fillna(0.0).astype("float32")
    g["Month_sin"] = pd.to_numeric(g["Month_sin"], errors="coerce").fillna(0.0).astype("float32")
    g["Month_cos"] = pd.to_numeric(g["Month_cos"], errors="coerce").fillna(0.0).astype("float32")
    g["CDD"] = pd.to_numeric(g["CDD"], errors="coerce").fillna(0.0).astype("float32")
    g["HDD"] = pd.to_numeric(g["HDD"], errors="coerce").fillna(0.0).astype("float32")
    g["wind_speed"] = pd.to_numeric(g["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature_lag_2"] = pd.to_numeric(g["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature_lag_6"] = pd.to_numeric(g["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature_lag_48"] = pd.to_numeric(g["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
    g["Temperature_lag_144"] = pd.to_numeric(g["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
    g["temperature_lag_2"] = pd.to_numeric(g["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
    g["temperature_lag_6"] = pd.to_numeric(g["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
    g["temperature_lag_48"] = pd.to_numeric(g["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
    g["temperature_lag_144"] = pd.to_numeric(g["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")

    g = g.reset_index()
    if len(g) < CONTEXT_LEN + SEQ_LEN:
        continue

    # Convert this customer's cleaned dataframe into arrays for the sliding-window loop.
    y_all = g[[TARGET_COL]].to_numpy(dtype=np.float32)
    x_time_past_all = g[PAST_COV_COLS].to_numpy(dtype=np.float32)
    x_time_future_all = g[FUTURE_COV_COLS].to_numpy(dtype=np.float32)
    x_static_vec = g[STATIC_COLS].iloc[0].to_numpy(dtype=np.float32)
    x_cat_vec = g[["StationNo_id", "TRIAL_REGION_NAME_id"]].iloc[0].to_numpy(dtype=np.int64)

    # Every window from this household goes to either train or validation, never both.
    is_val = household_id in val_customers
    yp_list = y_past_val_parts if is_val else y_past_parts
    yf_list = y_future_val_parts if is_val else y_future_parts
    xtp_list = x_time_past_val_parts if is_val else x_time_past_parts
    xtf_list = x_time_future_val_parts if is_val else x_time_future_parts
    xs_list = x_static_val_parts if is_val else x_static_parts
    xc_list = x_static_cat_val_parts if is_val else x_static_cat_parts

    # Slide the manually selected context/target window across this customer.
    max_start = len(g) - CONTEXT_LEN - SEQ_LEN
    for start in range(0, max_start + 1, WINDOW_STRIDE):
        mid = start + CONTEXT_LEN
        end = mid + SEQ_LEN
        yp_list.append(y_all[start:mid])
        yf_list.append(y_all[mid:end])
        xtp_list.append(x_time_past_all[start:mid])
        xtf_list.append(x_time_future_all[mid:end])
        xs_list.append(x_static_vec)
        xc_list.append(x_cat_vec)

# Stack collected lists into dense arrays with model-ready dtypes.
y_past = np.stack(y_past_parts).astype(np.float32)
y_future = np.stack(y_future_parts).astype(np.float32)
x_time_past = np.stack(x_time_past_parts).astype(np.float32)
x_time_future = np.stack(x_time_future_parts).astype(np.float32)
x_static = np.stack(x_static_parts).astype(np.float32)
static_cat_ids = np.stack(x_static_cat_parts).astype(np.int64)

y_past_val = np.stack(y_past_val_parts).astype(np.float32)
y_future_val = np.stack(y_future_val_parts).astype(np.float32)
x_time_past_val = np.stack(x_time_past_val_parts).astype(np.float32)
x_time_future_val = np.stack(x_time_future_val_parts).astype(np.float32)
x_static_val = np.stack(x_static_val_parts).astype(np.float32)
static_cat_ids_val = np.stack(x_static_cat_val_parts).astype(np.int64)

print("TRAIN y_past:", y_past.shape, "y_future:", y_future.shape, "x_time_past:", x_time_past.shape, "x_time_future:", x_time_future.shape)
print("VAL   y_past:", y_past_val.shape, "y_future:", y_future_val.shape, "x_time_past:", x_time_past_val.shape, "x_time_future:", x_time_future_val.shape)

# Fit target scaling on training kWh only, then reuse it for validation and inverse transforms.
y_scaler = MinMaxScaler(feature_range=(0, 1))
y_train_log_flat = np.concatenate([
    np.log1p(y_past).reshape(-1, 1),
    np.log1p(y_future).reshape(-1, 1),
], axis=0)
y_scaler.fit(y_train_log_flat)

def scale_y(arr):
    if np.asarray(arr).size == 0:
        return np.asarray(arr, dtype=np.float32)
    return y_scaler.transform(np.log1p(arr).reshape(-1, 1)).reshape(arr.shape).astype(np.float32)

def y_scaled_to_kwh(arr_scaled):
    arr_log = y_scaler.inverse_transform(np.asarray(arr_scaled).reshape(-1, 1)).reshape(np.asarray(arr_scaled).shape)
    return np.maximum(np.expm1(arr_log), 0.0).astype(np.float32)

y_past_scaled = scale_y(y_past)
y_future_scaled = scale_y(y_future)
y_past_scaled_val = scale_y(y_past_val)
y_future_scaled_val = scale_y(y_future_val)

y_dim = y_future_scaled.shape[-1]
past_c_dim = x_time_past.shape[-1]
future_c_dim = x_time_future.shape[-1]
s_dim = x_static.shape[-1]

# Fit past and future covariate scalers separately so unknowable weather never leaks into the future branch.
past_c_scaler = StandardScaler()
future_c_scaler = StandardScaler()

if x_time_past.size:
    past_c_scaler.fit(x_time_past.reshape(-1, past_c_dim))
if x_time_future.size:
    future_c_scaler.fit(x_time_future.reshape(-1, future_c_dim))

def scale_time_covariates(arr, scaler, dim):
    if arr.size == 0:
        return arr.astype(np.float32)
    return scaler.transform(arr.reshape(-1, dim)).reshape(arr.shape).astype(np.float32)

x_time_past_scaled = scale_time_covariates(x_time_past, past_c_scaler, past_c_dim)
x_time_past_scaled_val = scale_time_covariates(x_time_past_val, past_c_scaler, past_c_dim)
x_time_future_scaled = scale_time_covariates(x_time_future, future_c_scaler, future_c_dim)
x_time_future_scaled_val = scale_time_covariates(x_time_future_val, future_c_scaler, future_c_dim)

# Fit static numeric scaling on training windows only.
s_scaler = StandardScaler()
x_static_scaled = s_scaler.fit_transform(x_static).astype(np.float32)
x_static_scaled_val = s_scaler.transform(x_static_val).astype(np.float32)

if CONTEXT_LEN > 0:
    print("Scaled kWh range check:", y_past_scaled.min(), y_past_scaled.max(), y_future_scaled.min(), y_future_scaled.max())
else:
    print("Scaled kWh range check:", y_future_scaled.min(), y_future_scaled.max())
print("Model input dimensions:", "y_dim", y_dim, "past_c_dim", past_c_dim, "future_c_dim", future_c_dim, "s_dim", s_dim)


# Wrap the prepared NumPy windows in a PyTorch Dataset so the training loop receives consistent minibatches.
class ForecastWindowDataset(Dataset):
    def __init__(self, y_past, y_future, x_time_past, x_time_future, x_static, x_static_cat):
        self.y_past = torch.from_numpy(y_past).float()
        self.y_future = torch.from_numpy(y_future).float()
        self.x_time_past = torch.from_numpy(x_time_past).float()
        self.x_time_future = torch.from_numpy(x_time_future).float()
        self.x_static = torch.from_numpy(x_static).float()
        self.x_static_cat = torch.from_numpy(x_static_cat).long()

    def __len__(self):
        return self.y_future.shape[0]

    def __getitem__(self, idx):
        return (
            self.y_past[idx],
            self.y_future[idx],
            self.x_time_past[idx],
            self.x_time_future[idx],
            self.x_static[idx],
            self.x_static_cat[idx],
        )

train_dataset = ForecastWindowDataset(
    y_past_scaled,
    y_future_scaled,
    x_time_past_scaled,
    x_time_future_scaled,
    x_static_scaled,
    static_cat_ids,
)
loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=torch.cuda.is_available(),
)
print("Training windows:", len(train_dataset), "| batches per epoch:", len(loader))


In [ ]:

# RNN context encoder compresses past kWh plus past time covariates into one conditioning vector.
# Implementation reference for PyTorch RNN/LSTM sequence modelling: https://github.com/pytorch/examples/tree/main/word_language_model
class PastContextRNN(nn.Module):
    def __init__(self, y_dim, past_c_dim, hidden_dim):
        super().__init__()
        # PyTorch LSTM is the recurrent backbone for RNN sequence/context encoders; implementation: https://github.com/pytorch/pytorch/tree/main/torch/nn/modules/rnn.py
        self.rnn = nn.LSTM(y_dim + past_c_dim, hidden_dim, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, y_past, x_time_past):
        h, _ = self.rnn(torch.cat([y_past, x_time_past], dim=-1))
        return self.norm(h[:, -1])
# Combines future time covariates, static numeric features, categorical embeddings, and optional past context.
class FutureConditionProjector(nn.Module):
    def __init__(self, future_c_dim, s_num_dim, past_dim, cond_dim, num_stations, num_regions, station_emb_dim=16, region_emb_dim=4):
        super().__init__()
        self.station_emb = nn.Embedding(num_stations, station_emb_dim)
        self.region_emb = nn.Embedding(num_regions, region_emb_dim)
        in_dim = future_c_dim + s_num_dim + station_emb_dim + region_emb_dim + past_dim
        self.proj = nn.Sequential(
            nn.Linear(in_dim, cond_dim),
            nn.LayerNorm(cond_dim),
            nn.Tanh(),
        )

    def forward(self, x_time_future, x_static_num, x_static_cat, past_context):
        B, T, _ = x_time_future.shape
        station_id = x_static_cat[:, 0].long()
        region_id = x_static_cat[:, 1].long()
        st = self.station_emb(station_id).unsqueeze(1).expand(B, T, -1)
        rg = self.region_emb(region_id).unsqueeze(1).expand(B, T, -1)
        xs = x_static_num.unsqueeze(1).expand(B, T, -1)
        pc = past_context.unsqueeze(1).expand(B, T, -1)
        return self.proj(torch.cat([x_time_future, xs, st, rg, pc], dim=-1))
# Embeds the diffusion noise step, separate from the physical timestep inside the energy window; source: https://arxiv.org/abs/2006.11239
class TimeEmbedding(nn.Module):
    def __init__(self, steps, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(torch.arange(half, device=t.device, dtype=torch.float32) * -(math.log(10000.0) / max(1, half - 1)))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if emb.shape[1] < self.dim:
            emb = F.pad(emb, (0, self.dim - emb.shape[1]))
        return self.mlp(emb)


In [ ]:
# DDPM-style denoiser learns to predict injected Gaussian noise under conditions; source: https://arxiv.org/abs/2006.11239
# Implementation reference for DDPM training/sampling structure: https://github.com/openai/improved-diffusion
class ForecastDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.past = PastContextRNN(y_dim, past_c_dim, hidden_dim)
        self.cond = FutureConditionProjector(future_c_dim, s_dim, hidden_dim, cond_dim, num_stations, num_regions)
        self.time_emb = TimeEmbedding(diffusion_steps, hidden_dim)
        in_dim = y_dim + cond_dim + hidden_dim
        # PyTorch LSTM is the recurrent backbone for RNN sequence/context encoders; implementation: https://github.com/pytorch/pytorch/tree/main/torch/nn/modules/rnn.py
        self.net = nn.LSTM(in_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, y_dim)

    # Builds the conditional tensor reused by training, sampling, and evaluation.
    def build_cond(self, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        if y_past.shape[1] == 0:
            past_context = torch.zeros(
                y_past.shape[0], hidden_dim, device=y_past.device, dtype=y_past.dtype
            )
        else:
            past_context = self.past(y_past, x_time_past)
        return self.cond(x_time_future, x_static_num, x_static_cat, past_context)

    def forward(self, y_noisy, t, y_past, x_time_past, x_time_future, x_static_num, x_static_cat):
        cond = self.build_cond(y_past, x_time_past, x_time_future, x_static_num, x_static_cat)
        t_emb = self.time_emb(t).unsqueeze(1).expand(y_noisy.shape[0], y_noisy.shape[1], -1)
        h = torch.cat([y_noisy, cond, t_emb], dim=-1)
        h, _ = self.net(h)
        return self.out(h)


# Linear beta noise schedule follows the original DDPM setup; source: https://arxiv.org/abs/2006.11239
# Linear beta schedule mirrors common DDPM implementations; implementation reference: https://github.com/openai/improved-diffusion
betas = torch.linspace(beta_start, beta_end, diffusion_steps, device=DEVICE)
# alpha_t is the retained signal fraction at diffusion step t; source: https://arxiv.org/abs/2006.11239
alphas = 1.0 - betas
# alpha_bar_t is the cumulative retained signal used in the closed-form forward noising equation; source: https://arxiv.org/abs/2006.11239
# Cumulative alpha products implement the closed-form DDPM forward process; implementation reference: https://github.com/openai/improved-diffusion
alpha_bars = torch.cumprod(alphas, dim=0)

# Forward diffusion noising: y_t = sqrt(alpha_bar_t)y_0 + sqrt(1-alpha_bar_t)eps; source: https://arxiv.org/abs/2006.11239
# Implementation reference for DDPM forward noising utilities: https://github.com/openai/improved-diffusion
def q_sample(y0, t, noise):
    a_bar = alpha_bars[t].view(-1, 1, 1)
    return torch.sqrt(a_bar) * y0 + torch.sqrt(1.0 - a_bar) * noise

model_diff = ForecastDenoiser().to(DEVICE)
# AdamW is the decoupled-weight-decay optimiser used for neural training; implementation: https://github.com/pytorch/pytorch/tree/main/torch/optim
opt_diff = optim.AdamW(model_diff.parameters(), lr=LR, weight_decay=1e-4)

active_model = model_diff

# torch.no_grad disables autograd during sampling/evaluation to save memory; implementation: https://github.com/pytorch/pytorch
@torch.no_grad()
# Generates one batch of probabilistic forecasts in scaled space before kWh inversion.
def generate_one_batch(y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b):
    active_model.eval()
    y = torch.randn(y_past_b.shape[0], SEQ_LEN, y_dim, device=y_past_b.device) * sample_temperature
    for step in reversed(range(diffusion_steps)):
        t = torch.full((y.shape[0],), step, device=y.device, dtype=torch.long)
        eps = active_model(y, t, y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b)
        beta_t = betas[step]
        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]
        # DDPM reverse mean step removes predicted noise one schedule step at a time; source: https://arxiv.org/abs/2006.11239
        y = (1.0 / torch.sqrt(alpha_t)) * (y - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps)
        if step > 0:
            y = y + torch.sqrt(beta_t) * torch.randn_like(y) * sample_temperature
    return y.clamp(0.0, 1.0)


In [ ]:
# Train a new model. Run this cell before the save cell.
for epoch in range(epochs_diff):
    model_diff.train()
    running = 0.0
    for y_past_b, y_future_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b in loader:
        y_past_b = y_past_b.to(DEVICE, non_blocking=True)
        y_future_b = y_future_b.to(DEVICE, non_blocking=True)
        x_time_past_b = x_time_past_b.to(DEVICE, non_blocking=True)
        x_time_future_b = x_time_future_b.to(DEVICE, non_blocking=True)
        x_static_b = x_static_b.to(DEVICE, non_blocking=True)
        x_static_cat_b = x_static_cat_b.to(DEVICE, non_blocking=True).long()

        t = torch.randint(0, diffusion_steps, (y_future_b.shape[0],), device=DEVICE)
        noise = torch.randn_like(y_future_b)
        y_noisy = q_sample(y_future_b, t, noise)
        pred_noise = model_diff(y_noisy, t, y_past_b, x_time_past_b, x_time_future_b, x_static_b, x_static_cat_b)
        loss = F.mse_loss(pred_noise, noise)
        opt_diff.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip:
            # Gradient clipping is a standard stabilisation utility for RNN/Transformer/GAN training; implementation: https://github.com/pytorch/pytorch/tree/main/torch/nn/utils
            torch.nn.utils.clip_grad_norm_(model_diff.parameters(), grad_clip)
        opt_diff.step()
        running += loss.item()
    if epoch % 10 == 0:
        print(f"[{MODEL_LABEL}] epoch {epoch:03d} | mode={LOSS_MODE} | noise_mse={running/len(loader):.5f}")



In [ ]:
# Save the model produced by the training cell.
# Saves enough metadata with the weights to identify the exact experiment settings later.
def checkpoint_payload():
    extra = {
        "model_type": MODEL_LABEL,
        "model_kind": MODEL_KIND,
        "FREQ": FREQ,
        "PRIMARY_HORIZON": PRIMARY_HORIZON,
        "SEQ_LEN": SEQ_LEN,
        "CONTEXT_LEN": CONTEXT_LEN,
        "WINDOW_STRIDE": WINDOW_STRIDE,
        "PAST_COV_COLS": PAST_COV_COLS,
        "FUTURE_COV_COLS": FUTURE_COV_COLS,
        "STATIC_COLS": STATIC_COLS,
        "CATEGORICAL_COLS": CATEGORICAL_COLS,
        "LOSS_MODE": LOSS_MODE,
        "y_scaler_min": y_scaler.min_.tolist(),
        "y_scaler_scale": y_scaler.scale_.tolist(),
        "num_stations": num_stations,
        "num_regions": num_regions,
        "hidden_dim": hidden_dim,
        "cond_dim": cond_dim,
    }
    return extra
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_name = (
    f"{MODEL_TAG}__freq-{FREQ}__hor-{PRIMARY_HORIZON}__ctx-{CONTEXT_LEN}__seq-{SEQ_LEN}"
    f"__seed-{SEED}__mode-{LOSS_MODE}"
)
checkpoint_path = checkpoint_dir / f"{checkpoint_name}.pt"
payload = {"extra": checkpoint_payload(), "model_state_dict": model_diff.state_dict(), "optimizer_state_dict": opt_diff.state_dict()}
# torch.save serialises model weights plus metadata for reproducible checkpoint loading; implementation: https://github.com/pytorch/pytorch
torch.save(payload, checkpoint_path)
print("Saved:", checkpoint_path)


In [ ]:
# Select the checkpoint to load, then run this cell instead of the training and save cells.
CHECKPOINT_TO_LOAD = (
    Path("..") / "checkpoints" / "generative" /
    "forecast-rnn-diffusion__freq-30min__hor-24h__ctx-96__seq-48__seed-0__mode-core.pt"
)

# Loads a checkpoint saved by this notebook after checking shape-defining settings.
def load_checkpoint(path, *, load_optimizer=False):
    selected_path = Path(path)
    if not selected_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {selected_path}")

    checkpoint = torch.load(selected_path, map_location=DEVICE, weights_only=True)
    if "model_state_dict" not in checkpoint:
        raise KeyError("Checkpoint does not contain model_state_dict.")

    metadata = checkpoint.get("extra", {})
    expected = {
        "FREQ": FREQ,
        "PRIMARY_HORIZON": PRIMARY_HORIZON,
        "SEQ_LEN": SEQ_LEN,
        "CONTEXT_LEN": CONTEXT_LEN,
        "PAST_COV_COLS": PAST_COV_COLS,
        "FUTURE_COV_COLS": FUTURE_COV_COLS,
        "STATIC_COLS": STATIC_COLS,
        "CATEGORICAL_COLS": CATEGORICAL_COLS,
        "num_stations": num_stations,
        "num_regions": num_regions,
        "hidden_dim": hidden_dim,
        "cond_dim": cond_dim,
    }
    mismatches = {
        key: (metadata.get(key), expected_value)
        for key, expected_value in expected.items()
        if metadata.get(key) != expected_value
    }
    if mismatches:
        details = "; ".join(
            f"{key}: saved={saved!r}, current={current!r}"
            for key, (saved, current) in mismatches.items()
        )
        raise ValueError(f"Checkpoint settings do not match this run: {details}")

    saved_min = np.asarray(metadata.get("y_scaler_min"), dtype=np.float64)
    saved_scale = np.asarray(metadata.get("y_scaler_scale"), dtype=np.float64)
    if not (
        np.allclose(saved_min, y_scaler.min_, rtol=1e-6, atol=1e-7)
        and np.allclose(saved_scale, y_scaler.scale_, rtol=1e-6, atol=1e-7)
    ):
        raise ValueError("Checkpoint target scaling does not match the current training split.")

    model_diff.load_state_dict(checkpoint["model_state_dict"], strict=True)
    if load_optimizer:
        if "optimizer_state_dict" not in checkpoint:
            raise KeyError("Checkpoint does not contain optimizer_state_dict.")
        opt_diff.load_state_dict(checkpoint["optimizer_state_dict"])
    model_diff.to(DEVICE).eval()
    print("Loaded:", selected_path)
    return metadata

loaded_checkpoint_metadata = load_checkpoint(CHECKPOINT_TO_LOAD)
active_model = model_diff


In [ ]:
SAMPLES = 200
EVAL_BATCH_SIZE = 64
EVAL_WINDOWS = 512
EVAL_SEED = SEED + 10_123
BINS = 80

# The final evaluation uses q10, q50, and q90, giving an 80% interval.
LOWER_QUANTILE = 0.10
MEDIAN_QUANTILE = 0.50
UPPER_QUANTILE = 0.90
ALPHA_PI = 0.20

METRIC_NAMES = [
    "MAE",
    "RMSE",
    "PeakMAE",
    "QuantileLoss",
    "KL_Divergence",
    "DTW",
    "WinklerScore",
]


def pinball_loss(y_true, prediction, quantile):
    error = y_true - prediction
    return np.maximum(
        quantile * error,
        (quantile - 1.0) * error,
    )


def kl_divergence(real, forecast):
    real_flat = np.asarray(real, dtype=np.float32).reshape(-1)
    forecast_flat = np.asarray(forecast, dtype=np.float32).reshape(-1)
    lower = float(real_flat.min())
    upper = float(real_flat.max())
    if upper <= lower:
        upper = lower + 1e-6

    edges = np.linspace(lower, upper, BINS + 1)
    real_histogram, _ = np.histogram(real_flat, bins=edges)
    forecast_histogram, _ = np.histogram(forecast_flat, bins=edges)

    epsilon = 1e-8
    p = real_histogram.astype(np.float64) + epsilon
    q = forecast_histogram.astype(np.float64) + epsilon
    p /= p.sum()
    q /= q.sum()
    return float(np.sum(p * np.log(p / q)))


def dtw_distance(real, forecast):
    real = np.asarray(real, dtype=np.float32)
    forecast = np.asarray(forecast, dtype=np.float32)
    rows = len(real)
    columns = len(forecast)
    cost = np.full((rows + 1, columns + 1), np.inf, dtype=np.float32)
    cost[0, 0] = 0.0

    for row in range(1, rows + 1):
        for column in range(1, columns + 1):
            difference = abs(real[row - 1] - forecast[column - 1])
            cost[row, column] = difference + min(
                cost[row - 1, column],
                cost[row, column - 1],
                cost[row - 1, column - 1],
            )
    return float(cost[rows, columns])


def print_metrics_table(rows):
    table = pd.DataFrame(rows)
    table = table[METRIC_NAMES]
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(table.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
    return table


@torch.no_grad()
def evaluate_validation():
    rng = np.random.default_rng(EVAL_SEED)
    window_count = min(EVAL_WINDOWS, len(y_future_val))
    indices = rng.choice(len(y_future_val), size=window_count, replace=False)

    y_past_input = y_past_scaled_val[indices]
    past_covariate_input = x_time_past_scaled_val[indices]
    future_covariate_input = x_time_future_scaled_val[indices]
    static_input = x_static_scaled_val[indices]
    category_input = static_cat_ids_val[indices]
    y_real_kwh = y_future_val[indices]

    print(
        f"eval_pool=validation | windows={window_count} | "
        f"samples={SAMPLES}"
    )

    generated_scaled = []
    active_model.eval()
    for sample_index in range(SAMPLES):
        if sample_index % 10 == 0:
            print(
                f"Generating {MODEL_LABEL} sample "
                f"{sample_index + 1}/{SAMPLES}"
            )

        generated_batches = []
        for start in range(0, window_count, EVAL_BATCH_SIZE):
            end = min(start + EVAL_BATCH_SIZE, window_count)
            generated_batches.append(
                generate_one_batch(
                    torch.from_numpy(y_past_input[start:end]).to(DEVICE),
                    torch.from_numpy(past_covariate_input[start:end]).to(DEVICE),
                    torch.from_numpy(future_covariate_input[start:end]).to(DEVICE),
                    torch.from_numpy(static_input[start:end]).to(DEVICE),
                    torch.from_numpy(category_input[start:end]).to(DEVICE).long(),
                ).cpu().numpy()
            )
        generated_scaled.append(np.concatenate(generated_batches, axis=0))

    generated_scaled = np.stack(generated_scaled, axis=0)
    generated_kwh = np.stack(
        [
            y_scaled_to_kwh(generated_scaled[sample_index])
            for sample_index in range(SAMPLES)
        ],
        axis=0,
    )

    median_forecast = np.median(generated_kwh, axis=0)
    mean_forecast = np.mean(generated_kwh, axis=0, dtype=np.float64)

    MAE = float(np.mean(np.abs(median_forecast - y_real_kwh)))
    RMSE = float(np.sqrt(np.mean((mean_forecast - y_real_kwh) ** 2)))
    PeakMAE = float(
        np.mean(
            np.abs(
                median_forecast[:, :, 0].max(axis=1)
                - y_real_kwh[:, :, 0].max(axis=1)
            )
        )
    )
    KL_Divergence = kl_divergence(y_real_kwh, median_forecast)

    dtw_count = min(128, window_count)
    dtw_indices = rng.choice(window_count, size=dtw_count, replace=False)
    DTW = float(
        np.mean(
            [
                dtw_distance(
                    y_real_kwh[index, :, 0],
                    median_forecast[index, :, 0],
                )
                for index in dtw_indices
            ]
        )
    )

    sample_values = generated_kwh[:, :, :, 0]
    observed_values = y_real_kwh[:, :, 0]
    lower = np.quantile(sample_values, LOWER_QUANTILE, axis=0).astype(np.float32)
    median = np.quantile(sample_values, MEDIAN_QUANTILE, axis=0).astype(np.float32)
    upper = np.quantile(sample_values, UPPER_QUANTILE, axis=0).astype(np.float32)

    quantile_losses = [
        np.mean(pinball_loss(observed_values, lower, LOWER_QUANTILE)),
        np.mean(pinball_loss(observed_values, median, MEDIAN_QUANTILE)),
        np.mean(pinball_loss(observed_values, upper, UPPER_QUANTILE)),
    ]
    QuantileLoss = float(np.mean(quantile_losses))

    interval_width = upper - lower
    below = observed_values < lower
    above = observed_values > upper
    WinklerScore = float(
        np.mean(
            interval_width
            + (2.0 / ALPHA_PI) * (lower - observed_values) * below
            + (2.0 / ALPHA_PI) * (observed_values - upper) * above
        )
    )

    metrics = {
        "MAE": MAE,
        "RMSE": RMSE,
        "PeakMAE": PeakMAE,
        "QuantileLoss": QuantileLoss,
        "KL_Divergence": KL_Divergence,
        "DTW": DTW,
        "WinklerScore": WinklerScore,
    }

    print("=" * 60)
    print(
        f"[{MODEL_LABEL}] validation | freq={FREQ} | "
        f"horizon={PRIMARY_HORIZON} | context={CONTEXT_LEN}"
    )
    for metric_name in METRIC_NAMES:
        print(f"{metric_name}: {metrics[metric_name]:.6f}")
    return metrics


metrics_table = print_metrics_table([evaluate_validation()])
